# Step 2 – Data model (ERD) and the dataset split
Two questions before any analysis:
1. Are the tables linked correctly? (every key has a parent, no join duplicates rows)
2. How do we split the 23 tables into **business cards** (one question → one notebook → one dashboard page), like the Sharnoubi project?

In [1]:
import os
import pandas as pd

DATA = '../Cottonil_project/COTTONIL/COTTONIL_Data/02_TABLES'
t = {f[:-4]: pd.read_csv(os.path.join(DATA, f), low_memory=False) for f in sorted(os.listdir(DATA)) if f.endswith('.csv')}
print(len(t), 'files loaded')

24 files loaded


## 2.1 Check every relationship
A relationship = child table → parent table. Two checks (same as Sharnoubi):
- **parent key unique**: if a parent key repeats, every join doubles the child rows.
- **match %**: share of child keys that exist in the parent. A child row without a parent is an *orphan*.

In [2]:
locations = pd.concat([t['branches'].branch_id, t['warehouses'].warehouse_id])
REL = [  # (child table, child column, parent table, parent column)
    ('sales_orders', 'branch_id', 'branches', 'branch_id'),
    ('sales_orders', 'customer_id', 'customers', 'customer_id'),
    ('sales_order_items', 'sales_order_id', 'sales_orders', 'sales_order_id'),
    ('sales_order_items', 'sku_id', 'sku_variants', 'sku_id'),
    ('sales_order_items', 'product_id', 'products_catalog', 'product_id'),
    ('sku_variants', 'product_id', 'products_catalog', 'product_id'),
    ('sales_payments', 'sales_order_id', 'sales_orders', 'sales_order_id'),
    ('sales_returns', 'sales_line_id', 'sales_order_items', 'sales_line_id'),
    ('sales_returns', 'sku_id', 'sku_variants', 'sku_id'),
    ('delivery_shipments', 'sales_order_id', 'sales_orders', 'sales_order_id'),
    ('inventory_transfers', 'source_warehouse_id', 'warehouses', 'warehouse_id'),
    ('inventory_transfers', 'destination_branch_id', 'branches', 'branch_id'),
    ('inventory_transfer_items', 'transfer_id', 'inventory_transfers', 'transfer_id'),
    ('inventory_transfer_items', 'sku_id', 'sku_variants', 'sku_id'),
    ('purchase_orders', 'supplier_id', 'suppliers', 'supplier_id'),
    ('purchase_orders', 'receiving_warehouse_id', 'warehouses', 'warehouse_id'),
    ('purchase_order_items', 'po_id', 'purchase_orders', 'po_id'),
    ('purchase_order_items', 'sku_id', 'sku_variants', 'sku_id'),
    ('import_shipments', 'import_partner_id', 'import_partners', 'import_partner_id'),
    ('import_shipments', 'destination_warehouse_id', 'warehouses', 'warehouse_id'),
    ('employees', 'branch_id', 'branches', 'branch_id'),
    ('payroll', 'employee_id', 'employees', 'employee_id'),
]
rows = []
for child, ccol, parent, pcol in REL:
    keys = t[child][ccol].dropna()
    rows.append(dict(child=child, column=ccol, parent=parent, parent_key_unique=t[parent][pcol].is_unique,
                     match_pct=round(keys.isin(t[parent][pcol]).mean() * 100, 2), orphans=int((~keys.isin(t[parent][pcol])).sum()),
                     empty_keys=int(t[child][ccol].isna().sum())))
for child, ccol in [('monthly_stock', 'stock_location_id'), ('operating_expenses', 'branch_or_department_id')]:
    keys = t[child][ccol]
    rows.append(dict(child=child, column=ccol, parent='branches + warehouses', parent_key_unique=locations.is_unique,
                     match_pct=round(keys.isin(locations).mean() * 100, 2), orphans=int((~keys.isin(locations)).sum()), empty_keys=0))
rel = pd.DataFrame(rows)
print('relationships:', len(rel), '| all 100%:', (rel.match_pct == 100).all(), '| orphans:', rel.orphans.sum())
rel

relationships: 24 | all 100%: True | orphans: 0


,child,column,parent,parent_key_unique,match_pct,orphans,empty_keys
0,sales_orders,branch_id,branches,True,100.0,0,0
1,sales_orders,customer_id,customers,True,100.0,0,102961
2,sales_order_items,sales_order_id,sales_orders,True,100.0,0,0
3,sales_order_items,sku_id,sku_variants,True,100.0,0,0
4,sales_order_items,product_id,products_catalog,True,100.0,0,0
5,sku_variants,product_id,products_catalog,True,100.0,0,0
6,sales_payments,sales_order_id,sales_orders,True,100.0,0,0
7,sales_returns,sales_line_id,sales_order_items,True,100.0,0,0
8,sales_returns,sku_id,sku_variants,True,100.0,0,0
9,delivery_shipments,sales_order_id,sales_orders,True,100.0,0,0


**Result:** every key has a parent (0 orphans) and every parent key is unique.
The only empty keys are `customer_id` in `sales_orders` (orders without a customer – checked in card 03).
Two soft links have no key: campaigns link to sales only through the governorate name, and `factories` links to nothing.

In [3]:
# the join test: rows before = rows after (no duplicates, nothing lost)
li = t['sales_order_items']
j = (li.merge(t['sales_orders'][['sales_order_id', 'sales_channel', 'customer_id']], on='sales_order_id', how='left')
       .merge(t['sku_variants'][['sku_id', 'size']], on='sku_id', how='left')
       .merge(t['products_catalog'][['product_id', 'category_en']], on='product_id', how='left'))
print('before:', f'{len(li):,}', '| after:', f'{len(j):,}', '| same:', len(li) == len(j))
print('soft link – campaign governorates found in branches:',
      t['marketing_campaigns'].governorate_ar.isin(t['branches'].governorate_ar).mean().round(3))

before: 1,138,536 | after: 1,138,536 | same: True
soft link – campaign governorates found in branches: 1.0


## 2.2 Star schema (the model we use in Python, the dashboard and Power BI)
Facts = events with numbers and a date. Dimensions = descriptions with few rows. Every fact joins to dimensions by an ID.

In [4]:
star = pd.DataFrame([
    ('Fact', 'fact_sales_line', 'sales_order_items', 'one product line in an order', 'order_date, branch_id, sku_id, product_id, (customer_id via order)'),
    ('Fact', 'fact_order', 'sales_orders', 'one order', 'order_date, branch_id, customer_id'),
    ('Fact', 'fact_return', 'sales_returns', 'one returned line', 'return_date, sales_line_id, sku_id'),
    ('Fact', 'fact_shipment', 'delivery_shipments', 'one online delivery', 'ship_date, sales_order_id'),
    ('Fact', 'fact_payment', 'sales_payments', 'one payment', 'payment_date, sales_order_id'),
    ('Fact', 'fact_transfer_line', 'inventory_transfer_items + header', 'one SKU in a transfer', 'transfer_date, warehouse_id, branch_id, sku_id'),
    ('Fact', 'fact_po_line', 'purchase_order_items + header', 'one SKU in a purchase order', 'po_date, supplier_id, warehouse_id, sku_id'),
    ('Fact', 'fact_import', 'import_shipments', 'one import container', 'shipment_date, import_partner_id, warehouse_id'),
    ('Fact', 'fact_stock_month', 'monthly_stock', 'month × location × category', 'month, location_id, category'),
    ('Fact', 'fact_campaign', 'marketing_campaigns', 'one campaign', 'start_date, governorate'),
    ('Fact', 'fact_payroll', 'payroll', 'one payslip', 'pay_month, employee_id'),
    ('Fact', 'fact_expense', 'operating_expenses', 'one expense', 'expense_date, location_id'),
    ('Dimension', 'dim_date', '(generated)', 'one day, with fiscal year Oct–Sep', 'date'),
    ('Dimension', 'dim_product', 'products_catalog', 'one product', 'product_id'),
    ('Dimension', 'dim_sku', 'sku_variants', 'one size × colour', 'sku_id'),
    ('Dimension', 'dim_branch', 'branches', 'one branch / point of sale', 'branch_id'),
    ('Dimension', 'dim_customer', 'customers', 'one customer', 'customer_id'),
    ('Dimension', 'dim_warehouse', 'warehouses', 'one warehouse', 'warehouse_id'),
    ('Dimension', 'dim_supplier / dim_import_partner', 'suppliers, import_partners', 'one supplier / partner', 'supplier_id, import_partner_id'),
    ('Dimension', 'dim_employee', 'employees', 'one employee', 'employee_id'),
], columns=['type', 'model table', 'source', 'grain (one row =)', 'keys'])
star

,type,model table,source,grain (one row =),keys
0,Fact,fact_sales_line,sales_order_items,one product line in an order,"order_date, branch_id, sku_id, product_id, (customer_id via order)"
1,Fact,fact_order,sales_orders,one order,"order_date, branch_id, customer_id"
2,Fact,fact_return,sales_returns,one returned line,"return_date, sales_line_id, sku_id"
3,Fact,fact_shipment,delivery_shipments,one online delivery,"ship_date, sales_order_id"
4,Fact,fact_payment,sales_payments,one payment,"payment_date, sales_order_id"
5,Fact,fact_transfer_line,inventory_transfer_items + header,one SKU in a transfer,"transfer_date, warehouse_id, branch_id, sku_id"
6,Fact,fact_po_line,purchase_order_items + header,one SKU in a purchase order,"po_date, supplier_id, warehouse_id, sku_id"
7,Fact,fact_import,import_shipments,one import container,"shipment_date, import_partner_id, warehouse_id"
8,Fact,fact_stock_month,monthly_stock,month × location × category,"month, location_id, category"
9,Fact,fact_campaign,marketing_campaigns,one campaign,"start_date, governorate"


## 2.3 The dataset split – 23 tables → 9 business cards
Sharnoubi came with 10 case cards. Cottonil came with **22 app ideas** (the HVIA PDF, page 2–23).
So we built the split ourselves: each card = **one business question**, its tables, the HVIA ideas it answers,
its notebook and its dashboard page. Every table belongs to at least one card.

In [5]:
split = pd.DataFrame([
    ('01', 'Sales & growth', 'Sales grow every year – is it real growth or just prices?', 'sales_orders, sales_order_items, products_catalog, sku_variants',
     'Executive Dashboard · Sales Performance · Product Profitability', '04_card01_sales_growth', 'Executive / Sales'),
    ('02', 'Pricing & discounts', 'Do discounts sell more pieces, or just give away margin?', 'sales_order_items',
     'Price Optimization & Promotion', '05_card02_discounts', 'Sales & Pricing'),
    ('03', 'Customers', 'Who comes back, and can we trust the customer data?', 'customers, sales_orders',
     'Customer Segmentation & Loyalty · CLV & Churn', '06_card03_customers + 13_ai_churn_test', 'Customer 360'),
    ('04', 'Branches, channels & delivery', 'Which branch / channel creates value? Is delivery a problem?', 'branches, sales_orders, delivery_shipments, sales_returns, sales_payments',
     'Branch Performance · New Store Expansion · Omnichannel · Wholesale · Returns', '07_card04_branches_channels', 'Branches & Channels'),
    ('05', 'Inventory & allocation', 'Do branches get what they sell?', 'monthly_stock, inventory_transfers, inventory_transfer_items, warehouses',
     'Inventory & Warehouse · Smart Replenishment & Auto Allocation · Loss Prevention', '08_card05_inventory_allocation', 'Smart Inventory'),
    ('06', 'Purchasing & imports', 'Do we buy what we sell?', 'purchase_orders, purchase_order_items, suppliers, import_shipments, import_partners, factories',
     'Procurement & Supplier · Supply Chain & Logistics', '09_card06_purchasing', 'Procurement'),
    ('07', 'Marketing', 'Does the marketing money bring real orders?', 'marketing_campaigns (+ sales_orders by governorate)',
     'Marketing ROI & Campaign Performance', '10_card07_marketing', 'Marketing'),
    ('08', 'People & costs', 'Can we manage staff and costs with this data?', 'employees, payroll, operating_expenses',
     'Workforce Productivity & Store Operations', '11_card08_people_costs', 'People & Costs'),
    ('AI', 'Demand forecast', 'How much will we sell next quarter – per category and channel?', 'sales_order_items (+ forecast drives cards 05 & 06)',
     'Demand Forecasting & Product Planning · Smart Replenishment', '12_ai_demand_forecast', 'Smart Inventory'),
    ('09', 'The story & action plan', 'How much profit can the fixes bring, in EGP?', 'all cards',
     'Executive Dashboard', '14_card09_story_action_plan', 'Executive'),
    ('+', 'Data Trust Center (built without being asked)', 'Which of the 22 ideas can this data really support?', 'all tables + data_dictionary + public catalog',
     'all 22 ideas + AI Business Assistant', '15_app_data_trust', 'Data Trust Center'),
], columns=['card', 'name', 'business question', 'tables', 'HVIA ideas covered', 'notebook', 'dashboard page'])
used = set(', '.join(split.tables).replace('(', ',').replace(')', ',').replace('+', ',').split(','))
print('tables not covered by any card:', [k for k in t if k not in {u.strip() for u in used} and k != 'data_dictionary'])
split

tables not covered by any card: []


,card,name,business question,tables,HVIA ideas covered,notebook,dashboard page
0,01,Sales & growth,Sales grow every year – is it real growth or just prices?,"sales_orders, sales_order_items, products_catalog, sku_variants",Executive Dashboard · Sales Performance · Product Profitability,04_card01_sales_growth,Executive / Sales
1,02,Pricing & discounts,"Do discounts sell more pieces, or just give away margin?",sales_order_items,Price Optimization & Promotion,05_card02_discounts,Sales & Pricing
2,03,Customers,"Who comes back, and can we trust the customer data?","customers, sales_orders",Customer Segmentation & Loyalty · CLV & Churn,06_card03_customers + 13_ai_churn_test,Customer 360
3,04,"Branches, channels & delivery",Which branch / channel creates value? Is delivery a problem?,"branches, sales_orders, delivery_shipments, sales_returns, sales_payments",Branch Performance · New Store Expansion · Omnichannel · Wholesale · Returns,07_card04_branches_channels,Branches & Channels
4,05,Inventory & allocation,Do branches get what they sell?,"monthly_stock, inventory_transfers, inventory_transfer_items, warehouses",Inventory & Warehouse · Smart Replenishment & Auto Allocation · Loss Prevention,08_card05_inventory_allocation,Smart Inventory
5,06,Purchasing & imports,Do we buy what we sell?,"purchase_orders, purchase_order_items, suppliers, import_shipments, import_partners, factories",Procurement & Supplier · Supply Chain & Logistics,09_card06_purchasing,Procurement
6,07,Marketing,Does the marketing money bring real orders?,marketing_campaigns (+ sales_orders by governorate),Marketing ROI & Campaign Performance,10_card07_marketing,Marketing
7,08,People & costs,Can we manage staff and costs with this data?,"employees, payroll, operating_expenses",Workforce Productivity & Store Operations,11_card08_people_costs,People & Costs
8,AI,Demand forecast,How much will we sell next quarter – per category and channel?,sales_order_items (+ forecast drives cards 05 & 06),Demand Forecasting & Product Planning · Smart Replenishment,12_ai_demand_forecast,Smart Inventory
9,09,The story & action plan,"How much profit can the fixes bring, in EGP?",all cards,Executive Dashboard,14_card09_story_action_plan,Executive


**Takeaway:** the model is technically perfect (0 orphans, joins keep the row count).
The problems are not in the links, they are in the **content** of some columns – that is card by card from notebook 04.